# Direction 9: Query Rewrite Evaluation

This notebook compares the original `IntentExpander` with the optimized version for complex hotel-review queries.

Outputs:
- `query_rewrite_results.csv`: per-query rewrite results.
- `query_rewrite_summary.csv`: aggregate comparison metrics.


In [1]:
from pathlib import Path
import csv
import importlib.util
import json
import os
import sys
import time

import pandas as pd

BASE_DIR = Path.cwd()
if BASE_DIR.name != "task9_scripts_admit":
    BASE_DIR = Path(r"E:\研一下\LLM\final\task9_scripts_admit")

CSV_PATH = BASE_DIR / "complex_queries.csv"
BEFORE_INTENT = BASE_DIR / "before" / "intent.py"
AFTER_INTENT = BASE_DIR / "after" / "intent.py"
RESULT_PATH = BASE_DIR / "query_rewrite_results.csv"
SUMMARY_PATH = BASE_DIR / "query_rewrite_summary.csv"

print("BASE_DIR:", BASE_DIR)
print("CSV_PATH:", CSV_PATH)


BASE_DIR: e:\研一下\LLM\final\task9_scripts_admit
CSV_PATH: e:\研一下\LLM\final\task9_scripts_admit\complex_queries.csv


In [3]:
def load_env_file(path: Path):
    """Tiny .env loader to avoid depending on python-dotenv in this notebook."""
    if not path.exists():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)

load_env_file(BASE_DIR / ".env")

import dashscope
from dashscope import Generation


class NotebookLLMClient:
    """Minimal DashScope client for this task9 notebook."""

    def __init__(self, api_key: str, model: str = "qwen-flash", json: bool = True):
        self.api_key = api_key
        self.model = model
        self.json = json

    def generate(self, prompt: str, temperature: float = 0.7, json: bool | None = None) -> str:
        use_json = self.json if json is None else json
        response = Generation.call(
            api_key=self.api_key,
            model=self.model,
            prompt=prompt,
            temperature=temperature,
            result_format="message",
            response_format={"type": "json_object"} if use_json else None,
        )
        if response.status_code == 200:
            return response.output.choices[0].message.content.strip()
        raise RuntimeError(f"LLM call failed: {response.message}")

intl_key = os.getenv("DASHSCOPE_INTL_API_KEY")
api_key = intl_key or os.getenv("DASHSCOPE_API_KEY")
if not api_key:
    raise RuntimeError("Missing DASHSCOPE_API_KEY or DASHSCOPE_INTL_API_KEY. Please set it in .env or environment variables.")

if intl_key:
    dashscope.base_http_api_url = DASHSCOPE_INTL_API_BASE

MODEL = "qwen-flash"
llm_client = NotebookLLMClient(api_key, model=MODEL, json=True)
print("LLM model:", MODEL)
print("Using intl endpoint:", bool(intl_key))


LLM model: qwen-flash
Using intl endpoint: False


In [4]:
def load_module(path: Path, module_name: str):
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

before_module = load_module(BEFORE_INTENT, "task9_before_intent")
after_module = load_module(AFTER_INTENT, "task9_after_intent")

before_expander = before_module.IntentExpander(llm_client)
after_expander = after_module.IntentExpander(llm_client)

print("Loaded:", BEFORE_INTENT)
print("Loaded:", AFTER_INTENT)


Loaded: e:\研一下\LLM\final\task9_scripts_admit\before\intent.py
Loaded: e:\研一下\LLM\final\task9_scripts_admit\after\intent.py


In [5]:
queries_df = pd.read_csv(CSV_PATH, encoding="utf-8")
print("num queries:", len(queries_df))
display(queries_df.head())


num queries: 25


,qid,query,type,expected_intents
0,1,这家酒店适合带孩子住吗？交通和早餐怎么样？,多意图,亲子;交通;早餐
1,2,最近花园大床房隔音怎么样？晚上会不会吵？,约束+时效,花园大床房;隔音;最近;噪音
2,3,套房和普通房间差别大吗？值不值得升级？,比较型,套房;普通房;差异;升级;性价比
3,4,商务出差住这里方便吗？办公和交通条件怎么样？,场景型,商务;办公;交通
4,5,带老人入住会不会不方便？房间和服务怎么样？,人群场景,老人;房间;服务;便利性


## Run Rewrite Comparison

Set `MAX_ROWS` to a small number for a smoke test. Set it to `None` for the full 25-query evaluation.

In [6]:
MAX_ROWS = None  # Example: 3 for a quick smoke test; None for all rows.

INTENT_ALIASES = {
    "亲子": ["亲子", "孩子", "小孩", "儿童", "家庭"],
    "交通": ["交通", "出行", "地铁", "打车", "位置"],
    "早餐": ["早餐", "早饭", "餐饮"],
    "噪音": ["噪音", "吵", "隔音", "安静", "睡眠"],
    "性价比": ["性价比", "值不值", "值得", "划算", "价格"],
    "服务": ["服务", "前台", "响应", "接待"],
    "卫生": ["卫生", "干净", "清洁", "床品", "卫生间"],
    "商务": ["商务", "出差", "办公", "会议"],
    "设施": ["设施", "设备", "装修"],
    "空间": ["空间", "面积", "宽敞", "小不小", "局促"],
    "最近": ["最近", "现在", "最新", "当前"],
}

def aliases_for(label: str) -> list[str]:
    return INTENT_ALIASES.get(label, [label])

def coverage_score(expected_intents: str, rewritten_text: str) -> tuple[int, int, float, str]:
    labels = [x.strip() for x in str(expected_intents).split(";") if x.strip()]
    if not labels:
        return 0, 0, 0.0, ""
    matched = []
    for label in labels:
        if any(alias in rewritten_text for alias in aliases_for(label)):
            matched.append(label)
    return len(matched), len(labels), round(len(matched) / len(labels), 4), ";".join(matched)

def normalize_result(result):
    if not isinstance(result, list):
        return [], False, "result is not list"
    normalized = []
    for item in result:
        if not isinstance(item, dict):
            continue
        q = str(item.get("query", "")).strip()
        if not q:
            continue
        try:
            w = float(item.get("weight", 0))
        except Exception:
            w = 0.0
        normalized.append({"query": q, "weight": w})
    ok = 1 <= len(normalized) <= 3
    ok = ok and abs(sum(item["weight"] for item in normalized) - 1.0) < 1e-6
    return normalized, ok, "" if ok else "invalid count or weight sum"

def run_one(expander, query: str):
    start = time.time()
    try:
        raw = expander.expand(query)
        latency = time.time() - start
        normalized, format_ok, format_error = normalize_result(raw)
        return raw, normalized, format_ok, format_error, latency, ""
    except Exception as exc:
        latency = time.time() - start
        return None, [], False, "exception", latency, repr(exc)


In [7]:
eval_df = queries_df.head(MAX_ROWS) if MAX_ROWS else queries_df
records = []

for _, row in eval_df.iterrows():
    for version, expander in [("before", before_expander), ("after", after_expander)]:
        raw, normalized, format_ok, format_error, latency, error = run_one(expander, row["query"])
        rewritten_queries = [item["query"] for item in normalized]
        rewritten_weights = [item["weight"] for item in normalized]
        rewritten_text = " ".join(rewritten_queries)
        matched_count, expected_count, coverage, matched_labels = coverage_score(row["expected_intents"], rewritten_text)

        records.append({
            "qid": row["qid"],
            "version": version,
            "type": row["type"],
            "query": row["query"],
            "expected_intents": row["expected_intents"],
            "rewritten_queries": " ||| ".join(rewritten_queries),
            "weights": ";".join(str(w) for w in rewritten_weights),
            "num_rewritten_queries": len(rewritten_queries),
            "format_ok": format_ok,
            "format_error": format_error,
            "matched_intents": matched_labels,
            "matched_intent_count": matched_count,
            "expected_intent_count": expected_count,
            "intent_coverage": coverage,
            "latency_seconds": round(latency, 4),
            "error": error,
            "raw_result_json": json.dumps(raw, ensure_ascii=False),
        })
        print(f"{row['qid']} {version}: coverage={coverage}, format_ok={format_ok}, latency={latency:.2f}s")

results_df = pd.DataFrame(records)
results_df.to_csv(RESULT_PATH, index=False, encoding="utf-8-sig")
print("Saved:", RESULT_PATH)
display(results_df.head(10))


1 before: coverage=0.6667, format_ok=True, latency=0.97s
1 after: coverage=1.0, format_ok=True, latency=1.29s
2 before: coverage=0.75, format_ok=True, latency=1.03s
2 after: coverage=1.0, format_ok=True, latency=1.14s
3 before: coverage=1.0, format_ok=True, latency=1.39s
3 after: coverage=1.0, format_ok=True, latency=1.10s
4 before: coverage=1.0, format_ok=True, latency=1.31s
4 after: coverage=1.0, format_ok=True, latency=0.78s
5 before: coverage=0.75, format_ok=True, latency=1.23s
5 after: coverage=0.75, format_ok=True, latency=1.00s
6 before: coverage=0.3333, format_ok=True, latency=1.02s
6 after: coverage=0.3333, format_ok=True, latency=0.92s
7 before: coverage=0.0, format_ok=True, latency=0.80s
7 after: coverage=0.75, format_ok=True, latency=0.92s
8 before: coverage=1.0, format_ok=True, latency=0.91s
8 after: coverage=0.75, format_ok=True, latency=1.00s
9 before: coverage=0.6667, format_ok=True, latency=1.11s
9 after: coverage=0.6667, format_ok=True, latency=0.85s
10 before: covera

,qid,version,type,query,expected_intents,rewritten_queries,weights,num_rewritten_queries,format_ok,format_error,matched_intents,matched_intent_count,expected_intent_count,intent_coverage,latency_seconds,error,raw_result_json
0,1,before,多意图,这家酒店适合带孩子住吗？交通和早餐怎么样？,亲子;交通;早餐,酒店是否提供儿童友好设施？ ||| 酒店周边交通是否便利，是否有地铁或公交直达？,0.6;0.4,2,True,,亲子;交通,2,3,0.6667,0.9679,,"[{""query"": ""酒店是否提供儿童友好设施？"", ""weight"": 0.6}, {""..."
1,1,after,多意图,这家酒店适合带孩子住吗？交通和早餐怎么样？,亲子;交通;早餐,广州花园酒店是否适合带孩子入住？ ||| 广州花园酒店的交通便利性如何？ ||| 广州花园酒...,0.4;0.4;0.2,3,True,,亲子;交通;早餐,3,3,1.0000,1.2891,,"[{""query"": ""广州花园酒店是否适合带孩子入住？"", ""weight"": 0.4},..."
2,2,before,约束+时效,最近花园大床房隔音怎么样？晚上会不会吵？,花园大床房;隔音;最近;噪音,花园大床房的隔音效果如何，晚上是否有明显噪音干扰？ ||| 花园大床房在夜间是否容易受到外部...,0.8;0.2,2,True,,花园大床房;隔音;噪音,3,4,0.7500,1.0296,,"[{""query"": ""花园大床房的隔音效果如何，晚上是否有明显噪音干扰？"", ""weigh..."
3,2,after,约束+时效,最近花园大床房隔音怎么样？晚上会不会吵？,花园大床房;隔音;最近;噪音,最近花园大床房隔音效果如何？晚上是否吵闹？,1.0,1,True,,花园大床房;隔音;最近;噪音,4,4,1.0000,1.1358,,"[{""query"": ""最近花园大床房隔音效果如何？晚上是否吵闹？"", ""weight"": ..."
4,3,before,比较型,套房和普通房间差别大吗？值不值得升级？,套房;普通房;差异;升级;性价比,套房相比普通房间在面积和布局上有哪些具体差异？ ||| 套房的额外设施（如独立客厅、浴缸、厨...,0.6;0.4,2,True,,套房;普通房;差异;升级;性价比,5,5,1.0000,1.3944,,"[{""query"": ""套房相比普通房间在面积和布局上有哪些具体差异？"", ""weight""..."
5,3,after,比较型,套房和普通房间差别大吗？值不值得升级？,套房;普通房;差异;升级;性价比,广州花园酒店套房和普通房间在空间、设施和入住体验上有什么差异？ ||| 广州花园酒店套房相比...,0.8;0.2,2,True,,套房;普通房;差异;升级;性价比,5,5,1.0000,1.0984,,"[{""query"": ""广州花园酒店套房和普通房间在空间、设施和入住体验上有什么差异？"", ..."
6,4,before,场景型,商务出差住这里方便吗？办公和交通条件怎么样？,商务;办公;交通,酒店距离商务区和主要办公地点有多远？交通是否便利？ ||| 酒店是否提供适合商务出差的办公设...,0.6;0.4,2,True,,商务;办公;交通,3,3,1.0000,1.3059,,"[{""query"": ""酒店距离商务区和主要办公地点有多远？交通是否便利？"", ""weigh..."
7,4,after,场景型,商务出差住这里方便吗？办公和交通条件怎么样？,商务;办公;交通,广州花园酒店商务出差住在这里交通便利吗？ ||| 广州花园酒店为商务人士提供的办公设施条件如何？,0.6;0.4,2,True,,商务;办公;交通,3,3,1.0000,0.7833,,"[{""query"": ""广州花园酒店商务出差住在这里交通便利吗？"", ""weight"": 0..."
8,5,before,人群场景,带老人入住会不会不方便？房间和服务怎么样？,老人;房间;服务;便利性,酒店是否适合老人入住？有哪些无障碍设施或便利服务？ ||| 酒店房间的舒适度和布局是否适合老...,0.6;0.4,2,True,,老人;房间;服务,3,4,0.7500,1.2334,,"[{""query"": ""酒店是否适合老人入住？有哪些无障碍设施或便利服务？"", ""weigh..."
9,5,after,人群场景,带老人入住会不会不方便？房间和服务怎么样？,老人;房间;服务;便利性,带老人入住花园酒店是否方便？有哪些不便之处？ ||| 花园酒店的房间设施和服务对老年人友好吗？,0.6;0.4,2,True,,老人;房间;服务,3,4,0.7500,0.9970,,"[{""query"": ""带老人入住花园酒店是否方便？有哪些不便之处？"", ""weight"":..."


In [8]:
summary_df = (
    results_df.groupby("version", as_index=False)
    .agg(
        num_queries=("qid", "count"),
        parse_success_rate=("format_ok", "mean"),
        mean_intent_coverage=("intent_coverage", "mean"),
        mean_num_rewritten_queries=("num_rewritten_queries", "mean"),
        mean_latency_seconds=("latency_seconds", "mean"),
    )
)

for col in ["parse_success_rate", "mean_intent_coverage", "mean_num_rewritten_queries", "mean_latency_seconds"]:
    summary_df[col] = summary_df[col].round(4)

summary_df.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")
print("Saved:", SUMMARY_PATH)
display(summary_df)


Saved: e:\研一下\LLM\final\task9_scripts_admit\query_rewrite_summary.csv


,version,num_queries,parse_success_rate,mean_intent_coverage,mean_num_rewritten_queries,mean_latency_seconds
0,after,25,1.0,0.7153,2.04,1.0249
1,before,25,1.0,0.6807,2.00,1.0678


In [9]:
comparison_df = results_df.pivot_table(
    index=["qid", "type", "query", "expected_intents"],
    columns="version",
    values=["rewritten_queries", "weights", "intent_coverage", "format_ok"],
    aggfunc="first",
)
display(comparison_df)


format_ok         \
version                                                       after before   
qid type    query                      expected_intents                      
1   多意图     这家酒店适合带孩子住吗？交通和早餐怎么样？      亲子;交通;早餐                True   True   
2   约束+时效   最近花园大床房隔音怎么样？晚上会不会吵？       花园大床房;隔音;最近;噪音          True   True   
3   比较型     套房和普通房间差别大吗？值不值得升级？        套房;普通房;差异;升级;性价比        True   True   
4   场景型     商务出差住这里方便吗？办公和交通条件怎么样？     商务;办公;交通                True   True   
5   人群场景    带老人入住会不会不方便？房间和服务怎么样？      老人;房间;服务;便利性            True   True   
6   模糊体验型   酒店整体值不值得住？有没有明显缺点？         整体体验;推荐;缺点              True   True   
7   时效+多意图  最近服务质量怎么样？入住退房效率有没有问题？     最近;服务;入住效率;退房效率         True   True   
8   约束+多意图  花园双床房适合家庭住吗？房间空间和卫生怎么样？    花园双床房;家庭;空间;卫生          True   True   
9   多意图     早餐好不好吃？人多的时候会不会排队？         早餐;味道;排队                True   True   
10  多意图     酒店位置方便吗？周边吃饭和地铁情况怎么样？      位置;周边餐饮;地铁;交通           True   True   
11  多意图     房间设施是不是比较旧？但卫生和服务能不能接受？    房间设施;新旧;卫生;服务           True   True   
12  约束+人群   亲子主题房体验怎么样？孩子会喜欢吗？         亲子主题房;儿童;亲子体验           True   True   
13  多意图     行政酒廊和下午茶值得体验吗？服务好吗？        行政酒廊;下午茶;服务             True   True   
14  比较型     价格偏高吗？和酒店服务设施比起来值不值？       价格;服务;设施;性价比            True   True   
15  约束+人群   临街房间吵不吵？如果睡眠浅适合住吗？         临街房间;噪音;睡眠              True   True   
16  场景型     酒店适合度假吗？花园景观和餐饮体验怎么样？      度假;花园景观;餐饮              True   True   
17  约束+多意图  红棉大床套房住起来怎么样？空间、设施和服务都好吗？  红棉大床套房;空间;设施;服务         True   True   
18  时效+模糊体验 现在入住体验还好吗？老酒店设施会不会影响舒适度？   现在;入住体验;老酒店;设施;舒适度      True   True   
19  场景型     如果第一次来广州住这里合适吗？交通和周边方便吗？   第一次来广州;交通;周边配套          True   True   
20  模糊体验型   这家酒店有哪些优点和槽点？适合推荐给朋友吗？     优点;缺点;推荐                True   True   
21  场景型     停车方便吗？自驾入住会不会有额外麻烦？        停车;自驾;便利性               True   True   
22  人群场景    房间小不小？带两个孩子住会不会局促？         房间空间;亲子;家庭              True   True   
23  多意图     会员权益和升房体验怎么样？会不会有落差？       会员权益;升房;体验落差            True   True   
24  多意图     酒店卫生靠谱吗？卫生间和床品有没有被吐槽？      卫生;卫生间;床品;负面反馈          True   True   
25  场景型     这家酒店适合开会或接待客户吗？环境和服务够不够体面？ 会议;客户接待;环境;服务           True   True   

                                                          intent_coverage  \
version                                                             after   
qid type    query                      expected_intents                     
1   多意图     这家酒店适合带孩子住吗？交通和早餐怎么样？      亲子;交通;早餐                    1.0000   
2   约束+时效   最近花园大床房隔音怎么样？晚上会不会吵？       花园大床房;隔音;最近;噪音              1.0000   
3   比较型     套房和普通房间差别大吗？值不值得升级？        套房;普通房;差异;升级;性价比            1.0000   
4   场景型     商务出差住这里方便吗？办公和交通条件怎么样？     商务;办公;交通                    1.0000   
5   人群场景    带老人入住会不会不方便？房间和服务怎么样？      老人;房间;服务;便利性                0.7500   
6   模糊体验型   酒店整体值不值得住？有没有明显缺点？         整体体验;推荐;缺点                  0.3333   
7   时效+多意图  最近服务质量怎么样？入住退房效率有没有问题？     最近;服务;入住效率;退房效率             0.7500   
8   约束+多意图  花园双床房适合家庭住吗？房间空间和卫生怎么样？    花园双床房;家庭;空间;卫生              0.7500   
9   多意图     早餐好不好吃？人多的时候会不会排队？         早餐;味道;排队                    0.6667   
10  多意图     酒店位置方便吗？周边吃饭和地铁情况怎么样？      位置;周边餐饮;地铁;交通               0.5000   
11  多意图     房间设施是不是比较旧？但卫生和服务能不能接受？    房间设施;新旧;卫生;服务               0.7500   
12  约束+人群   亲子主题房体验怎么样？孩子会喜欢吗？         亲子主题房;儿童;亲子体验               0.3333   
13  多意图     行政酒廊和下午茶值得体验吗？服务好吗？        行政酒廊;下午茶;服务                 0.6667   
14  比较型     价格偏高吗？和酒店服务设施比起来值不值？       价格;服务;设施;性价比                1.0000   
15  约束+人群   临街房间吵不吵？如果睡眠浅适合住吗？         临街房间;噪音;睡眠                  1.0000   
16  场景型     酒店适合度假吗？花园景观和餐饮体验怎么样？      度假;花园景观;餐饮                  0.6667   
17  约束+多意图  红棉大床套房住起来怎么样？空间、设施和服务都好吗？  红棉大床套房;空间;设施;服务             1.0000   
18  时效+模糊体验 现在入住体验还好吗？老酒店设施会不会影响舒适度？   现在;入住体验;老酒店;设施;舒适度          0.8000   
19  场景型     如果第一次来广州住这里合适吗？交通和周边方便吗？   第一次来广州;交通;周边配套              0.6667   
20  模糊体验型   这家酒店有哪些优点和槽点？适合推荐给朋友吗？     优点;缺点;推荐                    0.3333   
21  场景型     停车方便吗？自驾入住会不会有额外麻烦？        停车;自驾;便利性                   0.6667   
22  人群场景    房间小不小？带两个孩子住会